# WP2: Out-of-Distribution Detection & Safe Mode

## Distributional Shift Robustness for Prometheus v0.97

**Prometheus v0.97** | [Open in Colab](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/master/notebooks/wp2_ood_detection_demo.ipynb)

This notebook demonstrates **Workplan 2: Address Distributional Shift**.

> *When a system encounters inputs significantly different from its training distribution,
> it should recognise that it cannot reliably act and escalate to human oversight.*

### Architecture

```
  Agent perceives state → Feature vector φ(s)
       ↓
  OODDetector.evaluate(φ)  ← trained on in-distribution data
       ↓
  SafeModeProtocol        OOD Score
       │                  ├── NORMAL    (score < 1.5σ)  → proceed normally
       │                  ├── CAUTION   (1.5–3.0σ)     → reduce confidence
       │                  ├── SAFE_MODE (3.0–6.0σ)     → restrict + alert
       │                  └── HALT      (> 6.0σ)       → refuse to act
       ↓
  MCSSupervisor.check_ood(φ) → SafeModeDecision
```

### Three detectors
| Detector | Method | Strength |
|---|---|---|
| `MahalanobisOODDetector` | Distance from fitted Gaussian | Fast, interpretable, handles correlations |
| `ReconstructionOODDetector` | Autoencoder reconstruction error | Catches non-Gaussian structure |
| `EnsembleOODDetector` | Weighted average of both | Most robust |

**Runtime**: ~2 minutes (CPU)

In [ ]:
# 1. Clone repo and install
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
    %cd Prometheus_v0_PoC
    !pip install -q numpy scipy matplotlib pytest
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

print('Setup complete')

## 1. Generate In-Distribution Training Data

In [ ]:
from benchmarks.ood_benchmark import (
    generate_in_distribution, generate_ood_mild, generate_ood_severe,
    gridworld_in_distribution, arc_in_distribution,
)
import numpy as np
import matplotlib.pyplot as plt

FEATURE_SIZE = 6
N_TRAIN      = 200

# Training data (GridWorld feature space)
train_features = gridworld_in_distribution(n=N_TRAIN, seed=42)

# Evaluation sets
eval_in_dist = gridworld_in_distribution(n=100, seed=99)            # same distribution
eval_mild    = generate_ood_mild(train_features, n=100, shift_sigma=3.0, seed=7)   # 3σ shift
eval_severe  = generate_ood_severe(FEATURE_SIZE, n=100, seed=13)    # far OOD

print('Dataset summary:')
print(f'  Training:        {train_features.shape}  (GridWorld in-distribution)')
print(f'  Eval in-dist:    {eval_in_dist.shape}')
print(f'  Eval mild OOD:   {eval_mild.shape}    (3σ feature shift)')
print(f'  Eval severe OOD: {eval_severe.shape}   (uniform far from origin)')
print()
print('Feature dimensions (GridWorld 6-D):')
feat_names = ['row_norm', 'col_norm', 'dist_to_goal', 'wall_clearance', 'at_goal', 'bias']
for i, (name, mean, std) in enumerate(
        zip(feat_names, train_features.mean(0), train_features.std(0))):
    print(f'  [{i}] {name:<18}  mean={mean:+.3f}  std={std:.3f}')

# Visualise 2-D projection
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, label, color in zip(
        axes,
        [eval_in_dist, eval_mild, eval_severe],
        ['In-distribution', 'Mild OOD (3σ)', 'Severe OOD'],
        ['#3498db', '#f39c12', '#e74c3c']):
    ax.scatter(train_features[:, 0], train_features[:, 1],
               c='lightgrey', s=15, alpha=0.5, label='Training')
    ax.scatter(data[:, 0], data[:, 1],
               c=color, s=20, alpha=0.7, label=label)
    ax.set_xlabel('row_norm'); ax.set_ylabel('col_norm')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Feature Space (2-D projection): Training vs Evaluation Sets',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Mahalanobis OOD Detector

In [ ]:
from prometheus.ood_detection import (
    MahalanobisOODDetector, ReconstructionOODDetector, EnsembleOODDetector,
    SafeModeProtocol, SafeMode, create_ood_detector
)

# Fit the Mahalanobis detector
maha = MahalanobisOODDetector(FEATURE_SIZE)
maha.fit(train_features)

print('Mahalanobis OOD Detector fitted')
print(f'  Training mean:       {maha._mean.round(3)}')
print(f'  Score calibration:   mean={maha._score_mean:.4f}  std={maha._score_std:.4f}')
print()

# Score a few examples
examples = [
    ('Normal GridWorld',  eval_in_dist[0]),
    ('Mild OOD',         eval_mild[0]),
    ('Severe OOD',       eval_severe[0]),
    ('All-zeros',        np.zeros(FEATURE_SIZE)),
    ('All-tens',         np.full(FEATURE_SIZE, 10.0)),
]

print(f'  {"Example":<25} {"Raw score":>11} {"Norm score":>11} {"Mode"}')
print('  ' + '-' * 60)
for name, x in examples:
    ood_obj = maha.evaluate(x)
    print(f'  {name:<25} {ood_obj.raw_score:>11.3f} {ood_obj.normalised:>11.3f}  {ood_obj.safe_mode.value}')

## 3. Score Distributions — All Three Sets

In [ ]:
def get_scores(detector, data):
    return np.array([detector.normalised_score(x) for x in data])

scores_in   = get_scores(maha, eval_in_dist)
scores_mild = get_scores(maha, eval_mild)
scores_sev  = get_scores(maha, eval_severe)

fig, ax = plt.subplots(figsize=(11, 4))
bins = np.linspace(0, max(scores_sev.max(), 15), 60)
ax.hist(scores_in,   bins=bins, alpha=0.6, color='#3498db', label='In-distribution', density=True)
ax.hist(scores_mild, bins=bins, alpha=0.6, color='#f39c12', label='Mild OOD (3σ)',   density=True)
ax.hist(scores_sev,  bins=bins, alpha=0.6, color='#e74c3c', label='Severe OOD',      density=True)

# Threshold lines
for thresh, label, color in [(1.5, 'CAUTION', 'orange'),
                               (3.0, 'SAFE_MODE', 'red'),
                               (6.0, 'HALT', 'darkred')]:
    ax.axvline(thresh, color=color, linestyle='--', linewidth=1.5, label=f'{label} threshold ({thresh}σ)')

ax.set_xlabel('Normalised OOD Score (Mahalanobis)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('OOD Score Distributions — Mahalanobis Detector',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0, min(bins[-1], 20))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Score statistics:')
for name, scores in [('In-distribution', scores_in),
                      ('Mild OOD',        scores_mild),
                      ('Severe OOD',      scores_sev)]:
    print(f'  {name:<20} mean={scores.mean():.2f}  std={scores.std():.2f}  '
          f'max={scores.max():.2f}  flagged(>2σ)={( scores>2.0).mean():.1%}')

## 4. Safe Mode Protocol — Escalating Response

In [ ]:
protocol = SafeModeProtocol(
    maha,
    low_threshold=1.5,
    high_threshold=3.0,
    critical_threshold=6.0,
)

print('Safe Mode Protocol evaluation:')
print()
test_cases = [
    ('Normal path (in-dist)',  eval_in_dist[0]),
    ('Slightly unusual',       eval_mild[5]),
    ('Moderately OOD',         eval_mild[0]),
    ('Severely OOD',           eval_severe[0]),
]
for desc, x in test_cases:
    decision = protocol.evaluate(x)
    allow    = 'ALLOW' if decision.allow_action else 'BLOCK'
    conf     = decision.confidence_scale
    alert    = '⚠ ALERT' if decision.alert_supervisor else ''
    print(f'  [{allow}] {desc:<28}  mode={decision.safe_mode.value:<10}  '
          f'confidence={conf:.1f}  {alert}')
    if decision.safe_mode != SafeMode.NORMAL:
        print(f'         → {decision.recommended_action[:80]}...')
    print()

print('\nProtocol summary (all evaluations so far):')
summary = protocol.summary()
for k, v in summary.items():
    if isinstance(v, dict):
        print(f'  {k}:')
        for kk, vv in v.items():
            print(f'    {kk}: {vv}')
    else:
        print(f'  {k}: {v}')

## 5. Reconstruction Autoencoder Detector

In [ ]:
print('Training ReconstructionOODDetector (autoencoder)...')
recon = ReconstructionOODDetector(FEATURE_SIZE, bottleneck_size=3, epochs=200)
recon.fit(train_features)
print(f'  Bottleneck size: {recon.bottleneck_size}')
print(f'  Score calibration: mean={recon._score_mean:.5f}  std={recon._score_std:.5f}')

r_in   = get_scores(recon, eval_in_dist)
r_mild = get_scores(recon, eval_mild)
r_sev  = get_scores(recon, eval_severe)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: score distributions
ax = axes[0]
bins = np.linspace(0, max(r_sev.max(), 6), 50)
ax.hist(r_in,   bins=bins, alpha=0.6, color='#3498db', label='In-distribution', density=True)
ax.hist(r_mild, bins=bins, alpha=0.6, color='#f39c12', label='Mild OOD',        density=True)
ax.hist(r_sev,  bins=bins, alpha=0.6, color='#e74c3c', label='Severe OOD',      density=True)
ax.axvline(2.0, color='red', linestyle='--', linewidth=1.5, label='Threshold (2σ)')
ax.set_xlabel('Normalised Reconstruction Error', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Reconstruction OOD Scores', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Right: Mahalanobis vs Reconstruction scatter
ax = axes[1]
m_in   = get_scores(maha, eval_in_dist)
m_mild = get_scores(maha, eval_mild)
m_sev  = get_scores(maha, eval_severe)
ax.scatter(m_in,   r_in,   c='#3498db', s=20, alpha=0.5, label='In-dist')
ax.scatter(m_mild, r_mild, c='#f39c12', s=20, alpha=0.5, label='Mild OOD')
ax.scatter(m_sev,  r_sev,  c='#e74c3c', s=20, alpha=0.5, label='Severe OOD')
ax.axvline(2.0, color='grey', linestyle='--', linewidth=1.0)
ax.axhline(2.0, color='grey', linestyle='--', linewidth=1.0)
ax.set_xlabel('Mahalanobis score (normalised)', fontsize=10)
ax.set_ylabel('Reconstruction score (normalised)', fontsize=10)
ax.set_title('Mahalanobis vs Reconstruction OOD Scores', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('OOD Detector Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Full Benchmark — AUROC, TPR, FPR

In [ ]:
from benchmarks.ood_benchmark import OODBenchmark

bench = OODBenchmark(
    feature_size=FEATURE_SIZE,
    n_train=200, n_in_dist=200,
    n_mild=100,  n_severe=100,
    seed=42,
)

detectors = [
    MahalanobisOODDetector(FEATURE_SIZE),
    ReconstructionOODDetector(FEATURE_SIZE, epochs=150),
    EnsembleOODDetector(FEATURE_SIZE, epochs=150),
]

results = bench.run_all(detectors, verbose=True)

# Visualise AUROC comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names  = [r.detector_name for r in results]
aurocs = [r.auroc for r in results]
tprs   = [r.tpr   for r in results]
fprs   = [r.fpr   for r in results]
colors = ['#2ecc71' if a > 0.8 else '#f39c12' if a > 0.6 else '#e74c3c' for a in aurocs]

for ax, vals, ylabel, title in [
    (axes[0], [a*100 for a in aurocs], 'AUROC (%)',         'AUROC (higher = better)'),
    (axes[1], [t*100 for t in tprs],   'TPR (%)',           'True Positive Rate'),
    (axes[2], [f*100 for f in fprs],   'FPR (%)',           'False Positive Rate (lower = better)'),
]:
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 115)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('OOD Detector Benchmark Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Integration with MCSSupervisor

In [ ]:
from prometheus.safety.mcs_supervisor import MCSSupervisor

# Fit the best detector
best_detector = EnsembleOODDetector(FEATURE_SIZE, epochs=100)
best_detector.fit(train_features)

# Create MCSSupervisor with OOD awareness
supervisor = MCSSupervisor(ood_detector=best_detector)
print(f'MCSSupervisor created')
print(f'  OOD detector attached: {supervisor.has_ood_detector}')
print()

# Simulate an agent loop that checks OOD before each action
print('Simulated agent loop (10 episodes):')
print(f'  {"Episode":<10} {"OOD Mode":<12} {"Allow?":<8} {"Confidence":<12} {"Alert?"}')
print('  ' + '-' * 55)

rng = np.random.default_rng(0)
for ep in range(10):
    # Mix of in-dist and OOD observations
    if ep < 6:
        obs = eval_in_dist[ep]                   # normal episodes
    elif ep < 8:
        obs = eval_mild[ep - 6]                  # mild OOD
    else:
        obs = eval_severe[ep - 8]                # severe OOD

    decision = supervisor.check_ood(obs)
    allow    = '✓' if decision.allow_action else '✗'
    alert    = '⚠' if decision.alert_supervisor else ''
    print(f'  Episode {ep+1:<3}  {decision.safe_mode.value:<12} {allow:<8} '
          f'{decision.confidence_scale:<12.1f} {alert}')

print()
print('OOD summary from supervisor:')
summary = supervisor.ood_summary()
print(f'  Total evaluations: {summary["total"]}')
print(f'  OOD rate:          {summary.get("ood_rate", 0):.1%}')
print(f'  Halt rate:         {summary.get("halt_rate", 0):.1%}')
print(f'  Blocked actions:   {summary.get("blocked", 0)}')

# Constitutional safety check still works
safe_code   = 'def add(x, y): return x + y'
unsafe_code = 'import subprocess\ndef run(cmd): return subprocess.call(cmd)'
c_ok  = supervisor.verify_modification(safe_code,   safe_code,   'math.py')
c_bad = supervisor.verify_modification(safe_code, unsafe_code, 'math.py')
print(f'\nConstitutional checks still operational:')
print(f'  Safe code:   is_safe={c_ok.is_safe}')
print(f'  Unsafe code: is_safe={c_bad.is_safe}, violation={c_bad.violation_type}')

## 8. Cross-Domain OOD: ARC vs OpenRA Features

In [ ]:
# Train on GridWorld features, then test with ARC-style features
# This simulates deploying an agent trained on one domain in a different domain

arc_feats  = arc_in_distribution(n=50, seed=0)   # 8-D, different structure

# Pad/truncate to FEATURE_SIZE for the demo
arc_feats_6d = arc_feats[:, :FEATURE_SIZE]

domain_examples = [
    ('GridWorld (trained domain)',  eval_in_dist[:20]),
    ('GridWorld OOD (mild)',        eval_mild[:20]),
    ('ARC features (wrong domain)', arc_feats_6d[:20]),
    ('Random noise (adversarial)',  np.random.randn(20, FEATURE_SIZE) * 5),
]

print('Cross-domain OOD scores (trained on GridWorld):')
print(f'  {"Domain":<35} {"Mean score":>12} {"OOD rate (>2σ)":>16}')
print('  ' + '-' * 65)

domain_means = []
domain_labels_plot = []
domain_colors_plot = ['#3498db', '#f39c12', '#9b59b6', '#e74c3c']

for (name, data), color in zip(domain_examples, domain_colors_plot):
    scores   = [maha.normalised_score(x) for x in data]
    ood_rate = (np.array(scores) > 2.0).mean()
    domain_means.append(np.mean(scores))
    domain_labels_plot.append(name)
    print(f'  {name:<35} {np.mean(scores):>12.3f} {ood_rate:>16.1%}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(domain_means)), domain_means,
              color=domain_colors_plot, edgecolor='black', linewidth=1.2)
ax.axhline(2.0, color='red', linestyle='--', linewidth=1.5, label='OOD threshold (2σ)')
ax.set_xticks(range(len(domain_labels_plot)))
ax.set_xticklabels(domain_labels_plot, rotation=15, ha='right')
ax.set_ylabel('Mean Normalised OOD Score', fontsize=11)
ax.set_title('Cross-Domain OOD Detection\n(detector trained on GridWorld)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, domain_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nKey insight: ARC and adversarial inputs are correctly flagged as OOD,')
print('even though they were never seen during training.')

## 9. Save / Load Detector

In [ ]:
import tempfile, json, os

# Save the Mahalanobis detector
with tempfile.NamedTemporaryFile(suffix='.json', delete=False) as f:
    save_path = f.name

maha.save(save_path)
file_size = os.path.getsize(save_path)
print(f'Saved to {save_path}  ({file_size} bytes)')

# Preview the JSON
with open(save_path) as f:
    saved = json.load(f)
print('Saved keys:', list(saved.keys()))

# Load and verify
loaded = MahalanobisOODDetector.load(save_path)
x_test = eval_in_dist[0]
orig_score   = maha.score(x_test)
loaded_score = loaded.score(x_test)
print(f'\nScore before save: {orig_score:.6f}')
print(f'Score after load:  {loaded_score:.6f}')
print(f'Match: {abs(orig_score - loaded_score) < 1e-9}')

## Summary

| Component | File | Purpose |
|---|---|---|
| `MahalanobisOODDetector` | `prometheus/ood_detection.py` | Fast, interpretable, no gradient descent |
| `ReconstructionOODDetector` | `prometheus/ood_detection.py` | Autoencoder; catches non-Gaussian OOD |
| `EnsembleOODDetector` | `prometheus/ood_detection.py` | Weighted combination; most robust |
| `SafeModeProtocol` | `prometheus/ood_detection.py` | NORMAL→CAUTION→SAFE_MODE→HALT escalation |
| `MCSSupervisor.check_ood()` | `prometheus/safety/mcs_supervisor.py` | Integrated OOD gate in the supervisor |
| `OODBenchmark` | `benchmarks/ood_benchmark.py` | AUROC/TPR/FPR evaluation framework |
| 66 unit tests | `tests/test_ood.py` | All passing ✅ |

### Safe Mode escalation
| Level | Score (σ) | Action | Confidence |
|---|---|---|---|
| NORMAL | < 1.5 | Proceed normally | 1.0 |
| CAUTION | 1.5 – 3.0 | Log warning, prefer conservative actions | 0.7 |
| SAFE_MODE | 3.0 – 6.0 | Restrict to pre-approved actions, alert supervisor | 0.3 |
| HALT | > 6.0 | Refuse to act, supervisor must review | 0.0 |